## Camada Silver

In [0]:
-- Salvando Silver apartir da Camada Bronze
CREATE TABLE `cat-adls-br-dev`.silver.ti_salaries
USING DELTA
LOCATION 'abfss://data@adlsdatalakehousedevbr.dfs.core.windows.net/silver/ti_salaries'
AS

WITH silver_base AS (
    SELECT
        work_year AS ano_pagamento,
        experience_level AS nivel_experiencia,
        employment_type AS tipo_contrato,
        job_title AS cargo,
        salary AS salario_bruto,
        salary_currency AS moeda_pagamento,
        salary_in_usd AS salario_usd,
        employee_residence AS pais_residencia,
        remote_ratio AS percentual_remoto,
        company_location AS pais_empresa,
        company_size AS tamanho_empresa
    FROM `cat-adls-br-dev`.bronze.ti_salaries
    WHERE work_year IS NOT NULL
),
silver_tamanho AS (
    SELECT
        ano_pagamento,
        nivel_experiencia,
        tipo_contrato,
        cargo,
        salario_bruto,
        moeda_pagamento,
        salario_usd,
        pais_residencia,
        percentual_remoto,
        pais_empresa,
        CASE
            WHEN tamanho_empresa = 'M' THEN 'Media'
            WHEN tamanho_empresa = 'L' THEN 'Grande'
            WHEN tamanho_empresa = 'S' THEN 'Pequena'
            ELSE tamanho_empresa
        END AS tamanho_empresa
    FROM silver_base
),
silver_experiencia AS (
    SELECT
        ano_pagamento,
        tipo_contrato,
        cargo,
        salario_bruto,
        moeda_pagamento,
        salario_usd,
        pais_residencia,
        percentual_remoto,
        pais_empresa,
        tamanho_empresa,
        CASE
            WHEN nivel_experiencia = 'EX' THEN 'Executivo'
            WHEN nivel_experiencia = 'MI' THEN 'Pleno'
            WHEN nivel_experiencia = 'EN' THEN 'Júnior'
            WHEN nivel_experiencia = 'SE' THEN 'Sênior'
            ELSE nivel_experiencia
        END AS nivel_experiencia
    FROM silver_tamanho
),
silver_contrato AS (
    SELECT
        ano_pagamento,
        nivel_experiencia,
        cargo,
        salario_bruto,
        moeda_pagamento,
        salario_usd,
        pais_residencia,
        percentual_remoto,
        pais_empresa,
        tamanho_empresa,
        CASE
            WHEN tipo_contrato = 'FT' THEN 'Tempo_Integral'
            WHEN tipo_contrato = 'PT' THEN 'Parcial'
            WHEN tipo_contrato = 'CP' THEN 'Contrato'
            WHEN tipo_contrato = 'FL' THEN 'Freelancer'
            ELSE tipo_contrato
        END AS tipo_contrato
    FROM silver_experiencia
),
silver_remoto AS (
    SELECT
        ano_pagamento,
        nivel_experiencia,
        tipo_contrato,
        cargo,
        salario_bruto,
        moeda_pagamento,
        salario_usd,
        pais_residencia,
        CASE
            WHEN CAST(percentual_remoto AS STRING) = '0' THEN 'Presencial'
            WHEN CAST(percentual_remoto AS STRING) = '50' THEN 'Hibrido'
            WHEN CAST(percentual_remoto AS STRING) = '100' THEN 'Remoto'
            ELSE CAST(percentual_remoto AS STRING)
        END AS percentual_remoto,
        pais_empresa,
        tamanho_empresa
    FROM silver_contrato
)
-- a saida desse select determinas as colunas a serem gravavada
SELECT *
FROM silver_remoto;

num_affected_rows,num_inserted_rows


## Salvando Dados da Silver na Camada Gold

In [0]:
CREATE TABLE `cat-adls-br-dev`.gold.ti_salaries 
USING DELTA
LOCATION 'abfss://data@adlsdatalakehousedevbr.dfs.core.windows.net/gold/ti_salaries'
AS 
SELECT *
FROM `cat-adls-br-dev`.silver.ti_salaries;

num_affected_rows,num_inserted_rows



## Fim